<h3>Data handling libraries</h3>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='darkgrid')

<h3>Importing models and metrics</h3>

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, auc, average_precision_score, roc_curve, precision_recall_curve, f1_score
from sklearn.cluster import Birch, KMeans, OPTICS

<h3>Loading main data</h3>

In [3]:
X_train_origin = pd.read_csv('../data/train_X.csv').fillna(0) 
final_test = pd.read_csv('../data/test1_X.csv').fillna(0)
y_train_origin = pd.read_csv('../data/train_y.csv')['default6']

In [4]:
X_train_origin = X_train_origin.drop(['report_date', 'contract_date'], axis=1)

<h3>Splitting for train and test</h3>

In [5]:
unique_contract_ids = X_train_origin['contract_id'].unique()
train_ids, test_ids = train_test_split(unique_contract_ids, test_size=0.2, random_state=42)

train_mask = X_train_origin['contract_id'].isin(train_ids)
test_mask = X_train_origin['contract_id'].isin(test_ids)

X_train = X_train_origin[train_mask] 
y_train = y_train_origin[train_mask]

X_test = X_train_origin[test_mask].drop(columns=['contract_id'])
y_test = y_train_origin[test_mask]


In [38]:
train_data = pd.concat([X_train, y_train], axis=1)
train_data.shape

(23192, 168)

In [39]:
# birch = Birch()
birch = Birch(n_clusters=6)
cluster_labels = birch.fit_predict(X_train.drop('contract_id', axis=1))
# train_data['cluster'] = cluster_labels

# clustered_data = {label: train_data[train_data['cluster'] == label].drop('cluster', axis=1)
#                   for label in np.unique(cluster_labels)}

# for cluster_label, data in clustered_data.items():
#     print(f"Cluster {cluster_label}: {data.shape[0]} samples")

# Example usage: Accessing data for a specific cluster
# cluster_0_data = clustered_data[0]


In [40]:
from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

class StackingModel(BaseEstimator):
    def __init__(self):
        # Инициализируем модели
        self.base_models = [
            ('catboost', CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=5,
                                            early_stopping_rounds=50, random_seed=42, verbose=0)),
            ('xgboost', XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=5,
                                      early_stopping_rounds=50, eval_metric='auc', random_state=42,
                                      use_label_encoder=False, verbosity=0)),
            ('lightgbm', lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.1, max_depth=5,
                                            early_stopping_rounds=50, random_state=42, verbosity=0)),
            ('randomforest', RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42))
        ]
        self.meta_model = LogisticRegression()

    def fit(self, X_train_base, y_train_base, X_meta, y_meta):
        meta_features = []  # Сохраняем мета-фичи 

        # Train each base model
        for name, model in self.base_models:
            if name in ['catboost', 'xgboost', 'lightgbm']:
                model.fit(X_train_base, y_train_base, eval_set=[(X_meta, y_meta)])
            else:
                model.fit(X_train_base, y_train_base)

            # Предсказываем мета-фичи
            meta_pred = model.predict_proba(X_meta)[:, 1]
            meta_features.append(meta_pred)

            # Считаем и выводим Gini score
            auc = roc_auc_score(y_meta, meta_pred)
            gini = 2 * auc - 1
            print(f'Base model {name} Gini Score on meta data: {gini:.4f}')

        # стакаем мета фичи и обучаем логрег
        self.meta_features_ = np.column_stack(meta_features)
        self.meta_model.fit(self.meta_features_, y_meta)

    def predict_proba(self, X_test):
        test_meta_features = []

        # Предсказываем вероятности с помощью base моделей
        for name, model in self.base_models:
            test_pred = model.predict_proba(X_test)[:, 1]
            test_meta_features.append(test_pred)

        test_meta_features = np.column_stack(test_meta_features)
        return self.meta_model.predict_proba(test_meta_features)[:, 1]

    def evaluate(self, X_test, y_test):
        # Считаем вероятности и метрику
        y_pred_proba = self.predict_proba(X_test)
        auc = roc_auc_score(y_test, y_pred_proba)
        gini = 2 * auc - 1
        print(f'Final Stacking Model Gini Score (No threshold): {gini:.4f}')
        return gini

# Использование
# Инициализируем модель
# stacking_model = StackingModel()

# Обучаем ее
# stacking_model.fit(X_train_base, y_train_base, X_meta, y_meta)

# Считаем метрику и фиксируем прибыль
# stacking_model.evaluate(X_test, y_test)


In [41]:
def get_cluster_models(train_data: pd.DataFrame, target_name: str, Clustering, Model):
    cluster_labels = Clustering.predict(train_data.drop(['contract_id', target_name], axis=1))
    train_data['cluster'] = cluster_labels
    clustered_data = {label: train_data[train_data['cluster'] == label].drop('cluster', axis=1)
                  for label in np.unique(cluster_labels)}
    cluster_models = {}
    for cluster_id, data in clustered_data.items():
        print(f'Processing and fitting model on the cluster №{cluster_id} ...')
        X = data.drop(target_name, axis=1)
        y = data[target_name]
        
        unique_contract_ids = X['contract_id'].unique()
        train_base_ids, meta_ids = train_test_split(unique_contract_ids, test_size=0.25, random_state=42)

        train_base_mask = X['contract_id'].isin(train_base_ids)
        meta_mask = X['contract_id'].isin(meta_ids)

        X_train_base = X[train_base_mask].drop(columns=['contract_id'])
        y_train_base = y[train_base_mask]

        X_meta = X[meta_mask].drop(columns=['contract_id'])
        y_meta = y[meta_mask]

        stacking_model = Model()
        stacking_model.fit(X_train_base, y_train_base, X_meta, y_meta)
        cluster_models[cluster_id] = stacking_model
        print('Stacking model fitted')
    return cluster_models

In [42]:
cluster_models = get_cluster_models(train_data, 'default6', birch, StackingModel)

Processing and fitting model on the cluster №0 ...
Base model catboost Gini Score on meta data: 0.1528
[0]	validation_0-auc:0.55214
[1]	validation_0-auc:0.57324
[2]	validation_0-auc:0.57795
[3]	validation_0-auc:0.57649
[4]	validation_0-auc:0.57326
[5]	validation_0-auc:0.58194
[6]	validation_0-auc:0.58390
[7]	validation_0-auc:0.60458
[8]	validation_0-auc:0.59258
[9]	validation_0-auc:0.59787
[10]	validation_0-auc:0.60159
[11]	validation_0-auc:0.61991
[12]	validation_0-auc:0.61624
[13]	validation_0-auc:0.62891
[14]	validation_0-auc:0.63367
[15]	validation_0-auc:0.62796
[16]	validation_0-auc:0.63149
[17]	validation_0-auc:0.63202
[18]	validation_0-auc:0.63160
[19]	validation_0-auc:0.63010
[20]	validation_0-auc:0.63446
[21]	validation_0-auc:0.63943
[22]	validation_0-auc:0.64200
[23]	validation_0-auc:0.64842
[24]	validation_0-auc:0.65321
[25]	validation_0-auc:0.64673
[26]	validation_0-auc:0.65682
[27]	validation_0-auc:0.65624
[28]	validation_0-auc:0.66041
[29]	validation_0-auc:0.65989
[30]	va

In [43]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
def create_submission(X_test_data: pd.DataFrame, Clustering, cluster_models):
    X_test = X_test_data.copy()
    cluster_ids = Clustering.predict(X_test)
    y_pred_proba = []
    for i in range(X_test.shape[0]):
        row, cluster_id = X_test.iloc[i, :].values.reshape(1,-1), cluster_ids[i]
        y_pred_proba.append(cluster_models[cluster_id].predict_proba(row)[0])
    return np.array(y_pred_proba)
        

In [44]:
y_pred_proba = create_submission(X_test, birch, cluster_models)

In [45]:
y_pred_proba

array([0.42709562, 0.14087818, 0.19392857, ..., 0.10225386, 0.10225386,
       0.11836132])

In [46]:
2 * roc_auc_score(y_test, y_pred_proba) - 1

0.5188704839741827

In [47]:
cluster_models

{0: StackingModel(),
 1: StackingModel(),
 2: StackingModel(),
 3: StackingModel(),
 4: StackingModel(),
 5: StackingModel()}